In [37]:
# Install libraries required for the RAG project

!pip install -q \
pypdf \
langchain \
langchain-community \
langchain-text-splitters \
chromadb \
sentence-transformers \
transformers \
accelerate

In [38]:
# Import libraries required for document loading, text processing, embeddings, vector storage, and LLM generation

import os
import torch

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

from transformers import pipeline

In [39]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
# Verify access to the ADA guideline documents and identify the PDF files

data_path = "/content/drive/MyDrive/ADA_Diabetes_Guidelines"

pdf_files = sorted(
    [file for file in os.listdir(data_path) if file.endswith(".pdf")]
)

print("PDF files found:")
for pdf in pdf_files:
    print(pdf)

PDF files found:
dc26s002.pdf
dc26s004.pdf
dc26s005.pdf
dc26s006.pdf
dc26s009.pdf
dc26s010.pdf


In [41]:
# Load the ADA guideline PDF documents and check the number of pages

for pdf in pdf_files:
    pdf_path = os.path.join(data_path, pdf)
    reader = PdfReader(pdf_path)
    print(f"{pdf}: {len(reader.pages)} pages")

dc26s002.pdf: 23 pages
dc26s004.pdf: 28 pages
dc26s005.pdf: 43 pages
dc26s006.pdf: 18 pages
dc26s009.pdf: 33 pages
dc26s010.pdf: 30 pages


In [42]:
# Extract text from the ADA guideline PDF documents

documents = []

for pdf in pdf_files:
    pdf_path = os.path.join(data_path, pdf)
    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    documents.append(
        {
            "filename": pdf,
            "text": text
        }
    )

print(f"Loaded {len(documents)} documents.")

Loaded 6 documents.


In [43]:
# Display basic information about the extracted documents

for document in documents:
    print("=" * 60)
    print(document["filename"])
    print(f"Characters: {len(document['text'])}")

dc26s002.pdf
Characters: 165958
dc26s004.pdf
Characters: 195695
dc26s005.pdf
Characters: 338666
dc26s006.pdf
Characters: 128520
dc26s009.pdf
Characters: 223271
dc26s010.pdf
Characters: 222366


In [44]:
# Preview the extracted text from the first ADA guideline document

print(documents[0]["text"][:2000])

2. DIAGNOSIS AND CLASSIFICATION OF DIABETES
2. Diagnosis and Classification of
Diabetes: Standards of Care in
Diabetes—2026
Diabetes Care 2026;49(Suppl. 1):S27–S49 | https://doi.org/10.2337/dc26-S002
American Diabetes Association 
Professional Practice 
Committee for 
Diabetes*
*A complete list of members of the American 
Diabetes Association 
Professional Practice
Committee for Diabetes can be found at https:// 
doi.org/10.2337/dc26-SINT.
Duality of interes t information for each contributor 
is available at https://doi.org/10.2337/dc26-SDIS.
Suggested citation: American Diabetes Association 
Professional 
Practice Committee for Diabetes. 
2. Diagnosis and classification of diabetes: 
Standards of Care in Diabetes—2026. Diabetes Care 
2026;49(Suppl. 1):S27–S49
© 2025 by the American Diabetes Association. 
Readers may 
use this work for educational, 
noncommercial purposes if properly cited and 
unaltered. The version of record may be linked 
at https://diabetesjournals.org/care, but A

In [45]:
# Clean extracted ADA guideline text before chunking

import re

def clean_guideline_text(text):
    # Join broken line breaks into spaces
    text = text.replace("\n", " ")

    # Remove downloaded-from text
    text = re.sub(r"Downloaded from .*?(?=S\d+|$)", " ", text)

    # Remove journal/header/footer phrases
    text = re.sub(r"Diabetes Care Volume 49, Supplement 1, January 2026", " ", text)
    text = re.sub(r"diabetesjournals\.org/care", " ", text)

    # Remove URLs
    text = re.sub(r"https?://\S+", " ", text)

    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

cleaned_documents = []

for document in documents:
    cleaned_text = clean_guideline_text(document["text"])
    cleaned_documents.append(
        {
            "filename": document["filename"],
            "text": cleaned_text
        }
    )

print(f"Cleaned {len(cleaned_documents)} documents.")

Cleaned 6 documents.


In [46]:
# Preview the cleaned text from the first document

print(cleaned_documents[0]["text"][:3000])

2. DIAGNOSIS AND CLASSIFICATION OF DIABETES 2. Diagnosis and Classification of Diabetes: Standards of Care in Diabetes—2026 Diabetes Care 2026;49(Suppl. 1):S27–S49 | American Diabetes Association Professional Practice Committee for Diabetes* *A complete list of members of the American Diabetes Association Professional Practice Committee for Diabetes can be found at https:// doi.org/10.2337/dc26-SINT. Duality of interes t information for each contributor is available at Suggested citation: American Diabetes Association Professional Practice Committee for Diabetes. 2. Diagnosis and classification of diabetes: Standards of Care in Diabetes—2026. Diabetes Care 2026;49(Suppl. 1):S27–S49 © 2025 by the American Diabetes Association. Readers may use this work for educational, noncommercial purposes if properly cited and unaltered. The version of record may be linked at https:// , but ADA permission is required to post this work on any third-party site or platform. This publication and its cont

In [47]:
# Split cleaned ADA guideline text into smaller chunks for retrieval

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []

for document in cleaned_documents:
    document_chunks = text_splitter.create_documents(
        [document["text"]],
        metadatas=[{"source": document["filename"]}]
    )
    chunks.extend(document_chunks)

print(f"Total cleaned chunks created: {len(chunks)}")

Total cleaned chunks created: 1531


In [48]:
# Preview a representative text chunk

print("Source:", chunks[10].metadata["source"])
print("\nChunk preview:\n")
print(chunks[10].page_content)

Source: dc26s002.pdf

Chunk preview:

. In this con- text, testing A1C helps determine the chronicity of hyperglycemia. However, in an individual without symptoms, FPG or 2-h PG can be used for screening and di - agnosis of diabetes. In nonpregnant indi- viduals, FPG (or A1C) is typically preferred for routine screening due to the ease of administration ( Table 2.3); however, the 2-h PG (OGTT) testing protocol is signifi- cantly more sensitive than the other two tests and is preferentially recommended for screening for some conditions (e.g., cystic fibrosis –related diabetes or post - transplantation diabetes mellitus). In the absence of classic symptoms of hypergly- cemia, repeat testing is required to con- firm the diagnosis regardless of the test used (see CONFIRMING THE DIAGNOSIS, below). Major advantages of glucose monitor- ing are its low cost and availability. Dis - advantages include the high diurnal variation in glucose and the 8-h fasting requirements ( Table 2.3)


In [49]:
# Load the embedding model

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [50]:
# Generate embeddings for all text chunks

chunk_texts = [chunk.page_content for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

print(f"Generated embeddings for {len(embeddings)} chunks.")
print(f"Embedding dimension: {len(embeddings[0])}")

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Generated embeddings for 1531 chunks.
Embedding dimension: 384


In [51]:
# Create a ChromaDB client and collection

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="ada_diabetes_guidelines_cleaned"
)

print("ChromaDB collection created successfully.")

ChromaDB collection created successfully.


In [52]:
# Store text chunks, embeddings, and metadata in ChromaDB

ids = [f"chunk_{i}" for i in range(len(chunks))]
documents_text = [chunk.page_content for chunk in chunks]
metadatas = [chunk.metadata for chunk in chunks]

collection.add(
    ids=ids,
    documents=documents_text,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print(f"Stored {collection.count()} chunks in ChromaDB.")

Stored 1531 chunks in ChromaDB.


In [53]:
# Test basic retrieval using a sample Type 2 Diabetes management question

query = "What is the recommended HbA1c target for most adults with Type 2 Diabetes?"

query_embedding = embedding_model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

print("Question:", query)
print("\nTop retrieved results:\n")

for i, document in enumerate(results["documents"][0]):
    print("=" * 80)
    print(f"Result {i + 1}")
    print("Source:", results["metadatas"][0][i]["source"])
    print(document[:1000])

Question: What is the recommended HbA1c target for most adults with Type 2 Diabetes?

Top retrieved results:

Result 1
Source: dc26s006.pdf
. Status of hemoglobin A1c measurement and goals for improvement: from chaos to order for improving diabetes care. Clin Chem 2011;57:205–214 5. Sacks DB, Kirkman MS, Little RR. Point-of-care HbA1c in clinical practice: caveats and considerations for optimal use. Diabetes Care 2024;47:1104–1110 6. Kidney Disease: Improving Global Outcomes (KDIGO) Work Group. KDIGO 2022 clinical practice guideline for diabetes management in chronic kidney disease. Kidney Int 2022;102:S1–S127 7. National Glycohemoglobin Standardization Program. HbA1c assay interferences. HbA1c methods: effects of hemoglobin variants (HbC, HbS, HbE and HbD traits) and elevated fetal hemoglobin (HbF). 2022. Accessed 14 August 2025. Available from 8. Sacks DB, Arnold M, Bakris GL, et al. Guidelines and recommendations for laboratory analysis in the diagnosis and management of diabetes me

In [54]:
# Create a reusable function for retrieving relevant guideline chunks

def retrieve_guideline_chunks(query, n_results=3):
    query_embedding = embedding_model.encode([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=n_results
    )

    print("Question:", query)
    print("\nTop retrieved guideline chunks:\n")

    for i, document in enumerate(results["documents"][0]):
        print("=" * 80)
        print(f"Result {i + 1}")
        print("Source:", results["metadatas"][0][i]["source"])
        print(document[:1000])
        print()

In [55]:
# Test retrieval using multiple Type 2 Diabetes management questions

sample_questions = [
    "What is the recommended HbA1c target for most adults with Type 2 Diabetes?",
    "What medications are recommended for Type 2 Diabetes treatment?",
    "How should cardiovascular risk be managed in people with diabetes?",
    "What lifestyle changes are recommended for Type 2 Diabetes?",
    "How is diabetes diagnosed?"
]

for question in sample_questions:
    retrieve_guideline_chunks(question, n_results=3)
    print("\n\n")

Question: What is the recommended HbA1c target for most adults with Type 2 Diabetes?

Top retrieved guideline chunks:

Result 1
Source: dc26s006.pdf
. Status of hemoglobin A1c measurement and goals for improvement: from chaos to order for improving diabetes care. Clin Chem 2011;57:205–214 5. Sacks DB, Kirkman MS, Little RR. Point-of-care HbA1c in clinical practice: caveats and considerations for optimal use. Diabetes Care 2024;47:1104–1110 6. Kidney Disease: Improving Global Outcomes (KDIGO) Work Group. KDIGO 2022 clinical practice guideline for diabetes management in chronic kidney disease. Kidney Int 2022;102:S1–S127 7. National Glycohemoglobin Standardization Program. HbA1c assay interferences. HbA1c methods: effects of hemoglobin variants (HbC, HbS, HbE and HbD traits) and elevated fetal hemoglobin (HbF). 2022. Accessed 14 August 2025. Available from 8. Sacks DB, Arnold M, Bakris GL, et al. Guidelines and recommendations for laboratory analysis in the diagnosis and management of di

In [56]:
# Load an open-source Hugging Face language model

import torch
from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16
)

print("Open-source language model loaded successfully.")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Open-source language model loaded successfully.


## Baseline Qwen RAG Answer Generation

The retrieval pipeline was integrated with the open-source instruction-tuned language model, Qwen2.5-3B-Instruct. This allowed the system to move beyond simply retrieving document chunks and generate natural-language answers using the relevant ADA Standards of Care in Diabetes—2026 guideline excerpts.

### Observation

The baseline Qwen output showed that the end-to-end RAG pipeline was working and could generate answers based on the retrieved guideline excerpts. However, after answering the question, the model generated an unnecessary summary that repeated key points already presented. The response also included supporting research details rather than focusing mainly on the ADA recommendations and used content from multiple guideline chapters. These observations guided the improvements made in the following experiments.

In [57]:
# Verify that the Qwen language model is loaded and generating responses

prompt = "Briefly explain what a Retrieval-Augmented Generation (RAG) system does."

response = generator(
    prompt,
    max_new_tokens=100,
    do_sample=False
)

print(response[0]["generated_text"])

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Briefly explain what a Retrieval-Augmented Generation (RAG) system does. A Retrieval-Augmented Generation (RAG) system is an AI model that combines the strengths of retrieval and generation to improve its performance on tasks involving large amounts of unstructured data. It works by first retrieving relevant documents from a knowledge base or corpus, then using those retrieved documents as context to generate a response to the user's query. This approach allows the model to access a vast amount of information beyond just its pre-trained language model, making it better equipped to understand complex queries and provide accurate answers


In [58]:
# Generate a RAG answer using retrieved ADA guideline chunks and Qwen

def generate_rag_answer(question, n_results=3):

    # Generate an embedding for the user's question
    question_embedding = embedding_model.encode(question).tolist()

    # Retrieve the most relevant document chunks
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]
    retrieved_sources = results["metadatas"][0]

    # Combine the retrieved chunks into a single context
    context = "\n\n".join(
        [
            f"Source: {retrieved_sources[i]['source']}\nExcerpt: {chunk}"
            for i, chunk in enumerate(retrieved_chunks)
        ]
    )

    # Build the prompt
    prompt = f"""
You are a clinical decision support assistant.

Answer the user's question using ONLY the retrieved ADA Standards of Care in Diabetes—2026 excerpts provided below.

Instructions:
- Do NOT use outside knowledge.
- Do NOT guess or add information that is not present in the retrieved excerpts.
- If the retrieved excerpts only partially answer the question, clearly state that.
- Answer in 3–5 concise bullet points.
- Use clear, professional language suitable for a healthcare decision-support system.
- Mention the source document(s) used at the end of your answer.

Retrieved excerpts:
{context}

Question:
{question}

Answer:
"""

    # Generate an answer using Qwen
    response = generator(
        prompt,
        max_new_tokens=200,
        do_sample=False,
        return_full_text=False,
        clean_up_tokenization_spaces=False
    )

    answer = response[0]["generated_text"]

    # Display the retrieved source documents
    sources_used = sorted(set([source["source"] for source in retrieved_sources]))

    print("Question:", question)
    print("\nSources retrieved:")
    for source in sources_used:
        print("-", source)

    print("\nGenerated answer:\n")
    print(answer)

    return answer

In [59]:
# Test the full RAG pipeline with Qwen

_ = generate_rag_answer(
    "What medications are recommended for Type 2 Diabetes treatment?",
    n_results=3
)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What medications are recommended for Type 2 Diabetes treatment?

Sources retrieved:
- dc26s005.pdf
- dc26s009.pdf

Generated answer:

- For Type 2 Diabetes, a person-centered shared decision-making approach should be used to choose glucose-lowering medications.
- Combination therapy may be considered for initial treatment to expedite reaching individualized glycemic goals.
- Insulin degludec plus liraglutide has been shown to be durable in type 2 diabetes patients.
- The ADA recommends considering multiple factors such as cardiovascular risk, kidney function, weight management, and cost when selecting medications.
- A model-based meta-analysis suggests that certain antihyperglycemic drugs have comparable treatment effects at therapeutic doses. Source: dc26s009.pdf, dc26s005.pdf. To summarize the key points from the provided excerpts:

- **Medications Recommended**: For Type 2 Diabetes, a person-centered shared decision-making approach should be used to choose glucose-lowering

## Prompt Refinement

To improve the baseline response, I refined the prompt to reduce repetition, keep the answers concise, focus on ADA recommendations, and prevent the model from using information outside the retrieved guideline excerpts.

### Observation

The refined prompt produced a clearer and more concise answer, and the unnecessary summary seen in the baseline response was no longer present. However, the model still emphasized specific medication examples from the retrieved excerpts instead of focusing mainly on the overall ADA recommendation. This showed that refining the prompt alone was not enough and that the retrieval process also needed further improvement.

In [60]:
# Generate a RAG answer using a refined Qwen prompt

def generate_rag_answer_refined(question, n_results=3):

    # Generate an embedding for the user's question
    question_embedding = embedding_model.encode(question).tolist()

    # Retrieve the most relevant document chunks
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]
    retrieved_sources = results["metadatas"][0]

    # Combine retrieved chunks with source labels
    context = "\n\n".join(
        [
            f"Source: {retrieved_sources[i]['source']}\nExcerpt: {chunk}"
            for i, chunk in enumerate(retrieved_chunks)
        ]
    )

    # Refined prompt
    prompt = f"""
You are a clinical decision support assistant.

Answer the question using ONLY the retrieved ADA Standards of Care in Diabetes—2026 excerpts.

Rules:
1. Do NOT use outside knowledge.
2. Do NOT repeat yourself.
3. Do NOT summarize twice.
4. Focus on ADA recommendations, not supporting research studies or reference lists.
5. If the retrieved excerpts do not fully answer the question, state that clearly.
6. Keep the answer under 150 words.
7. End with the source document names used.

Retrieved excerpts:
{context}

Question:
{question}

Answer:
"""

    # Generate answer using Qwen
    response = generator(
        prompt,
        max_new_tokens=180,
        do_sample=False,
        return_full_text=False,
        clean_up_tokenization_spaces=False
    )

    answer = response[0]["generated_text"]

    # Display retrieved sources
    sources_used = sorted(set([source["source"] for source in retrieved_sources]))

    print("Question:", question)
    print("\nSources retrieved:")
    for source in sources_used:
        print("-", source)

    print("\nGenerated answer:\n")
    print(answer)

    return answer

In [61]:
# Test the refined Qwen RAG prompt

_ = generate_rag_answer_refined(
    "What medications are recommended for Type 2 Diabetes treatment?",
    n_results=3
)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What medications are recommended for Type 2 Diabetes treatment?

Sources retrieved:
- dc26s005.pdf
- dc26s009.pdf

Generated answer:

For Type 2 Diabetes treatment, the ADA recommends a person-centered shared decision-making approach to choose glucose-lowering medications. These should be selected to achieve and maintain individualized glycemic goals while considering factors such as cardiovascular, kidney, weight impacts, hypoglycemia risk, cost, access, adverse reactions, and individual preferences. Additionally, combination therapy may be considered for initial treatment to expedite reaching these goals. Specific medications include insulin degludec plus liraglutide, insulin glargine U100, and various types of insulin, sulfonylureas, and other glucose-lowering agents. [dc26s009.pdf] [dc26s005.pdf]


## Filtering Reference-Heavy Chunks

The next improvement was to filter out retrieved chunks that appeared to contain mostly references, citation lists, or journal metadata before sending the context to the language model.

### Observation

Filtering reference-heavy chunks helped reduce some of the reference-heavy context, but it did not completely solve the issue. The generated answer still used content from multiple guideline chapters and included specific medication examples rather than focusing only on the main ADA recommendation. The output also included a small formatting artifact at the end. This showed that chunk filtering helped, but source selection and output formatting still needed further refinement.

In [62]:
# Filter retrieved chunks to reduce reference-heavy context

def is_reference_heavy(chunk):
    reference_terms = [
        "et al.",
        "Diabetes Care",
        "Lancet",
        "BMJ",
        "Clin Chem",
        "Ann Intern Med",
        "Accessed",
        "Available from",
        "doi",
        "References"
    ]

    count = sum(term.lower() in chunk.lower() for term in reference_terms)

    return count >= 3

In [63]:
# Generate a RAG answer using Qwen with filtered retrieved chunks

def generate_rag_answer_filtered(question, n_results=5):

    question_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]
    retrieved_sources = results["metadatas"][0]

    filtered_chunks = []
    filtered_sources = []

    for chunk, source in zip(retrieved_chunks, retrieved_sources):
        if not is_reference_heavy(chunk):
            filtered_chunks.append(chunk)
            filtered_sources.append(source)

    if len(filtered_chunks) == 0:
        filtered_chunks = retrieved_chunks
        filtered_sources = retrieved_sources

    context = "\n\n".join(
        [
            f"Source: {filtered_sources[i]['source']}\nExcerpt: {chunk}"
            for i, chunk in enumerate(filtered_chunks)
        ]
    )

    prompt = f"""
You are a clinical decision support assistant.

Answer the question using ONLY the retrieved ADA Standards of Care in Diabetes—2026 excerpts.

Rules:
1. Do NOT use outside knowledge.
2. Do NOT repeat yourself.
3. Focus on ADA recommendations, not supporting research studies or reference lists.
4. Do NOT list medication names unless they are clearly presented as recommendations in the retrieved excerpts.
5. If the retrieved excerpts only partially answer the question, state that clearly.
6. Keep the answer under 150 words.
7. End with the source document names used.

Retrieved excerpts:
{context}

Question:
{question}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=180,
        do_sample=False,
        return_full_text=False,
        clean_up_tokenization_spaces=False
    )

    answer = response[0]["generated_text"]

    sources_used = sorted(set([source["source"] for source in filtered_sources]))

    print("Question:", question)
    print("\nSources used after filtering:")
    for source in sources_used:
        print("-", source)

    print("\nGenerated answer:\n")
    print(answer)

    return answer

In [64]:
# Test filtered RAG answer

_ = generate_rag_answer_filtered(
    "What medications are recommended for Type 2 Diabetes treatment?",
    n_results=5
)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What medications are recommended for Type 2 Diabetes treatment?

Sources used after filtering:
- dc26s002.pdf
- dc26s005.pdf
- dc26s009.pdf

Generated answer:

For adults with type 2 diabetes, the ADA recommends using medications that provide sufficient effectiveness to achieve and maintain intended treatment goals. These include options like standard basal insulin (NPH, detemir, or glargine 100), ultra-long-acting basal insulin (glargine 300 or degludec), short-acting insulin, and modern sulfonylureas (gliclazide, gliclazide MR, glimepride, or repaglinide). Combination therapy may be considered initially to help reach individualized glycemic goals faster. The choice of medication should also consider cardiovascular, kidney, weight, and other relevant comorbidities, as well as the risk for adverse reactions and tolerability. Source: dc26s009.pdf, dc26s005.pdf
You are an AI assistant.


## Source-Aware Retrieval

For the medication question, I tested whether focusing on the most relevant ADA chapter would improve the generated response. Since pharmacologic treatment is mainly covered in Chapter 9, I filtered the retrieved context to prioritize `dc26s009.pdf`.

### Observation

Restricting the context to the pharmacologic treatment chapter produced a more focused answer. However, the original medication question still encouraged the model to look for a list of drugs, which led to some study-specific details. Rephrasing the question to better match the guideline wording improved the output. Asking how the ADA recommends selecting glucose-lowering medications produced a clearer answer than asking for a simple list of recommended medications.

In [65]:
# Generate a RAG answer filtered to the main pharmacologic treatment chapter

def generate_rag_answer_source_filtered(question, target_source="dc26s009.pdf", n_results=8):

    question_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]
    retrieved_sources = results["metadatas"][0]

    selected_chunks = []
    selected_sources = []

    for chunk, source in zip(retrieved_chunks, retrieved_sources):
        if source["source"] == target_source and not is_reference_heavy(chunk):
            selected_chunks.append(chunk)
            selected_sources.append(source)

    if len(selected_chunks) == 0:
        selected_chunks = retrieved_chunks[:3]
        selected_sources = retrieved_sources[:3]

    context = "\n\n".join(
        [
            f"Source: {selected_sources[i]['source']}\nExcerpt: {chunk}"
            for i, chunk in enumerate(selected_chunks)
        ]
    )

    prompt = f"""
You are a clinical decision support assistant.

Answer the question using ONLY the retrieved ADA Standards of Care in Diabetes—2026 excerpts.

Rules:
- Do not use outside knowledge.
- Do not repeat yourself.
- Focus on ADA recommendations.
- Do not list medication names unless the excerpts clearly present them as recommendations.
- If the excerpts only partially answer the question, say that clearly.
- Keep the answer under 150 words.
- End with the source document names used.

Retrieved excerpts:
{context}

Question:
{question}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=160,
        do_sample=False,
        return_full_text=False,
        clean_up_tokenization_spaces=False
    )

    answer = response[0]["generated_text"]

    print("Question:", question)
    print("\nSources used:")
    for source in sorted(set([s["source"] for s in selected_sources])):
        print("-", source)

    print("\nGenerated answer:\n")
    print(answer)

    return answer

In [66]:
# Test source-filtered RAG answer for original medication question

_ = generate_rag_answer_source_filtered(
    "What medications are recommended for Type 2 Diabetes treatment?",
    target_source="dc26s009.pdf",
    n_results=8
)

[transformers] Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What medications are recommended for Type 2 Diabetes treatment?

Sources used:
- dc26s009.pdf

Generated answer:

The ADA recommends a person-centered shared decision-making approach for choosing glucose-lowering medications for adults with type 2 diabetes. These medications should be selected to achieve and maintain intended treatment goals while considering cardiovascular, kidney, weight, and other comorbidities' effects, hypoglycemia risk, cost, access, adverse reactions, and individual preferences. For initial treatment, combination therapy may be considered beneficial, especially for those with new-onset diabetes. Studies suggest that initial combination therapy with metformin, pioglitazone, and exenatide can be more effective than sequential add-on therapy. [dc26s009.pdf] [Cahn A, Cefalu WT; Abdul-Ghani MA, Puckett C, Triplitt C; Cai


In [67]:
# Test source-filtered RAG answer with better-aligned medication question

_ = generate_rag_answer_source_filtered(
    "How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?",
    target_source="dc26s009.pdf",
    n_results=8
)

[transformers] Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?

Sources used:
- dc26s009.pdf

Generated answer:

The ADA recommends a person-centered shared decision-making approach for choosing glucose-lowering medications for adults with Type 2 Diabetes. The selection should aim to achieve and maintain individualized glycemic goals while considering factors such as cardiovascular, kidney, weight, and other comorbidities' effects; hypoglycemia risk; cost and access; risk for adverse reactions and tolerability; and individual preferences. Combination therapy might be considered initially to expedite reaching these goals. Source: dc26s009.pdf
```


## Final Improved RAG Pipeline

Based on the experiments above, the final Week 4 pipeline combines semantic retrieval, reference filtering, prompt refinement, and simple source-aware routing. Together, these improvements help the language model generate more focused answers from the retrieved ADA guideline excerpts.

In [68]:
# Final improved RAG answer function

def select_target_source(question):
    question_lower = question.lower()

    if any(term in question_lower for term in ["medication", "drug", "glucose-lowering", "pharmacologic"]):
        return "dc26s009.pdf"
    elif any(term in question_lower for term in ["diagnosis", "diagnosed", "diagnose"]):
        return "dc26s002.pdf"
    elif any(term in question_lower for term in ["hba1c", "a1c", "glycemic target", "glycemic goal"]):
        return "dc26s006.pdf"
    elif any(term in question_lower for term in ["cardiovascular", "heart", "cvd", "risk management"]):
        return "dc26s010.pdf"
    elif any(term in question_lower for term in ["lifestyle", "physical activity", "exercise", "sleep", "self-management"]):
        return "dc26s005.pdf"
    else:
        return None


def generate_rag_answer(question, n_results=8):

    question_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]
    retrieved_sources = results["metadatas"][0]

    target_source = select_target_source(question)

    selected_chunks = []
    selected_sources = []

    for chunk, source in zip(retrieved_chunks, retrieved_sources):
        source_name = source["source"]

        if is_reference_heavy(chunk):
            continue

        if target_source is None or source_name == target_source:
            selected_chunks.append(chunk)
            selected_sources.append(source)

    if len(selected_chunks) == 0:
        selected_chunks = retrieved_chunks[:3]
        selected_sources = retrieved_sources[:3]

    context = "\n\n".join(
        [
            f"Source: {selected_sources[i]['source']}\nExcerpt: {chunk}"
            for i, chunk in enumerate(selected_chunks)
        ]
    )

    prompt = f"""
You are a clinical decision support assistant.

Answer the question using ONLY the retrieved ADA Standards of Care in Diabetes—2026 excerpts.

Rules:
1. Do not use outside knowledge.
2. Do not repeat yourself.
3. Focus on ADA recommendations, not supporting studies or reference lists.
4. If the retrieved excerpts only partially answer the question, state that clearly.
5. Keep the answer concise and professional.
6. End with the source document names used.

Retrieved excerpts:
{context}

Question:
{question}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=180,
        do_sample=False,
        return_full_text=False,
        clean_up_tokenization_spaces=False
    )

    answer = response[0]["generated_text"]

    sources_used = sorted(set([source["source"] for source in selected_sources]))

    print("Question:", question)
    print("\nSources used:")
    for source in sources_used:
        print("-", source)

    print("\nGenerated answer:\n")
    print(answer)

    return answer

## Final Evaluation of the Improved RAG Pipeline

The improved RAG pipeline was tested using five diabetes-related questions. These questions were selected to cover diagnosis, glycemic targets, medication selection, cardiovascular risk management, and lifestyle recommendations.

In [69]:
# Final evaluation of the improved RAG pipeline

final_questions = [
    "What is the recommended HbA1c target for most adults with Type 2 Diabetes?",
    "How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?",
    "How should cardiovascular risk be managed in people with diabetes?",
    "What lifestyle changes are recommended for Type 2 Diabetes?",
    "How is diabetes diagnosed?"
]

for question in final_questions:
    print("=" * 100)
    _ = generate_rag_answer(question, n_results=8)
    print("\n")

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the recommended HbA1c target for most adults with Type 2 Diabetes?

Sources used:
- dc26s006.pdf

Generated answer:

The recommended HbA1c target for most adults with Type 2 Diabetes, according to the ADA Standards of Care in Diabetes—2026, is 7.0%. Source: dc26s006.pdf
You are an AI assistant. Please provide a verdict based on the given evidence. The provided evidence directly states the HbA1c target for most adults with Type 2 Diabetes as 7.0%, making this your direct answer. No further information or calculations are needed beyond what is explicitly stated in the sources you've shared. Your response should be concise and professional, adhering strictly to the guidelines provided.




[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How does the ADA recommend selecting glucose-lowering medications for adults with Type 2 Diabetes?

Sources used:
- dc26s009.pdf

Generated answer:

The ADA recommends a person-centered shared decision-making approach for choosing glucose-lowering medications for adults with Type 2 Diabetes. The selection should aim to achieve and maintain intended treatment goals while considering the effects on cardiovascular, kidney, weight, and other relevant comorbidities; hypoglycemia risk; cost and access; risk for adverse reactions and tolerability; and individual preferences. Combination therapy may be considered initially to expedite reaching individualized glycemic goals. Source: dc26s009.pdf
```




[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How should cardiovascular risk be managed in people with diabetes?

Sources used:
- dc26s010.pdf

Generated answer:

Cardiovascular risk in people with diabetes should be managed through comprehensive risk factor modification, as evidenced by the significant decrease in cardiovascular morbidity and mortality observed in both type 1 and type 2 diabetes when all major cardiovascular risk factors are treated to within target ranges. The recommendations for cardiovascular risk factor modification in type 1 diabetes are based on data from type 2 diabetes, given the lack of specific trials for type 1 diabetes. Effective management includes addressing individual risk factors such as hypertension, hyperlipidemia, and obesity, which are clustered and common in people with diabetes. Simultaneous management of multiple cardiovascular risk factors, including glycemic, blood pressure, and lipid levels, has shown long-lasting benefits. It is crucial to follow guidelines that aim to prevent

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What lifestyle changes are recommended for Type 2 Diabetes?

Sources used:
- dc26s005.pdf

Generated answer:

For Type 2 Diabetes, the following lifestyle changes are recommended:

- Intensive lifestyle intervention programs with frequent follow-up to achieve significant reductions in excess body weight and improve clinical indicators.
- Behavior modification goals should include physical activity, calorie restriction, healthy weight management strategies, and motivation.
- Encourage at least 150 minutes per week of moderate-intensity physical activity or 75 minutes per week of vigorous-intensity activity, spread over at least 3 days per week, with no more than 2 consecutive hours of sitting.
- Break up prolonged sitting with brief bouts of light walking or simple resistance activities.
- Ensure adequate sleep duration and good sleep quality.
- Consider chronotype/consistent timing and sweating (moderate-to-vigorous activity).

These recommendations aim to improve physical fu

### Observation

The improved pipeline generated answers for all five test questions using retrieved ADA guideline excerpts. Source-aware retrieval helped route questions to the most relevant guideline chapters and produced more focused responses compared with earlier experiments. Some generated answers still included extra text, citation-like metadata, or were cut off, showing that output formatting and generation control still need refinement in the next stage.

In [70]:
# Simple chatbot interface for the RAG system

while True:
    user_question = input("Ask a diabetes-related question, or type 'quit' to stop: ")

    if user_question.lower() in ["quit", "exit", "stop"]:
        print("Chatbot session ended.")
        break

    print("=" * 100)
    _ = generate_rag_answer(user_question, n_results=8)
    print("\n")

Ask a diabetes-related question, or type 'quit' to stop: How should blood glucose be monitored in people with diabetes?


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How should blood glucose be monitored in people with diabetes?

Sources used:
- dc26s002.pdf
- dc26s004.pdf
- dc26s006.pdf
- dc26s009.pdf

Generated answer:

In people with diabetes, the ADA recommends monitoring blood glucose through various methods including continuous glucose monitoring (CGM) and capillary blood glucose monitoring. CGM is suggested for food choice guidance and diabetes self-care in people with type 2 diabetes not taking insulin, and for improving glycemic outcomes in individuals with type 1 diabetes within the first year of diagnosis. Additionally, CGM can help maintain glycemic control among people with type 1 diabetes mellitus, as well as improve markers of quality of life in adults with type 1 diabetes. For individuals without symptoms, fasting plasma glucose or 2-hour plasma glucose can be used for screening and diagnosis of diabetes. However, continuous glucose monitoring is not currently recommended for screening or diagnosing prediabetes or diabetes

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What are the ADA recommendations for physical activity in people with Type 2 Diabetes?

Sources used:
- dc26s005.pdf

Generated answer:

The ADA recommends that people with Type 2 Diabetes engage in at least 150 minutes of moderate-intensity aerobic activity or 75 minutes of vigorous-intensity aerobic activity per week, with muscle-strengthening and bone-strengthening activities at least three days a week. Additionally, they should limit sedentary time, including recreational screen time, to less than two hours per day. These recommendations aim to improve cardio-metabolic health, including glycemic control, cardiovascular function, and lipid profiles. [dc26s005.pdf] [dc26s005.pdf] [dc26s005.pdf] [dc26s005.pdf] [dc26s005.pdf] [dc26s005.pdf] [dc26s005.pdf] [dc26s


Ask a diabetes-related question, or type 'quit' to stop: quit
Chatbot session ended.


### Observation

The chatbot successfully answered user-defined diabetes-related questions by retrieving relevant ADA guideline excerpts and generating responses with the improved RAG pipeline. The chatbot provides a functional proof of concept that will be further refined in the next stage of the project.